In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data import SliceConditionDataset
from models import CustomVAE, DDPM, SliceConditionedTimeUNet
from training import Trainer
from training.loss import SliceConditionedDiffusionLoss

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
PATCH_SIZE = 64
BATCH_SIZE = 2
TIMESTEPS = 10
AXIS = "z"
SLICE_INDEX = 12
SEED = 0


In [2]:
dataset = SliceConditionDataset(DATA_DIR, patch_size=PATCH_SIZE, axis=AXIS, slice_index=SLICE_INDEX, seed=SEED)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
batch = next(iter(loader))
batch["target"].shape, batch["condition"].shape, batch["axis"], batch["slice_index"]


(torch.Size([2, 1, 64, 64]),
 torch.Size([2, 1, 64, 64]),
 tensor([2, 2]),
 tensor([12, 12]))

In [3]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])

unet = SliceConditionedTimeUNet(latent_ch=4, base_ch=16, time_dim=16, max_slices=64)
ddpm = DDPM(timesteps=TIMESTEPS)
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda _: 1.0)
criterion = SliceConditionedDiffusionLoss(vae=vae, ddpm=ddpm)
trainer = Trainer(
    model=unet,
    train_loader=loader,
    valid_loader=None,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    save_dir=ROOT / "output" / "notebook_train_step",
)


In [4]:
losses = trainer.step()
losses


1.0845654010772705